Part F — Technical Requirements
Question 38
Use Spark transformations wherever practical.

Include examples of:

select
withColumn
when
otherwise
cast
trim
upper/lower/initcap
regexp_replace
to_date
coalesce
join
groupBy
agg
countDistinct
sum
avg
row_number
rank or dense_rank
Window
Question 39
Avoid collecting the full source dataset to the driver.

Small reference sheets may be converted through Python when necessary, but the appointment transformations and aggregations must be performed using Spark.

Question 40
Make the pipeline rerunnable.

A second execution should not create uncontrolled duplicate data.

Document whether you use:

overwrite
append with deduplication
run-specific paths
Question 41
Add Markdown documentation before each major layer:

RAW
BRONZE
SILVER
GOLD
Explain:

Purpose
Input
Output
Main transformations
Validation performed
Question 42
At the end of the notebook, print or display a control summary:

Raw files copied
Bronze rows
Silver clean rows
Silver rejected rows
Gold tables created
Reconciliation passed
Pipeline status
Deliverables
Submit:

Databricks notebook exported as .dbc, .ipynb, or source file.
Screenshot of RAW folder.
Screenshot of Bronze outputs.
Screenshot of Silver clean and rejected counts.
Screenshot of Gold department report.
Screenshot of Gold doctor ranking.
Screenshot of the data-quality report.
A short document containing the ten business insights.
A brief explanation of the medallion architecture used.
The final reconciliation result.

Databricks Medallion Architecture Assignment
Owen Peterson
7/18/26
#TODO


In [1]:
import os

from pyspark.sql.types import IntegerType, DecimalType, StructType, DoubleType

base_directory: str = os.getcwd()
hospital_medallion_path: str = os.path.join(base_directory, "Volumes/tables/medallion_hospital")
input_path: str = os.path.join(hospital_medallion_path, "input")
raw_path: str = os.path.join(hospital_medallion_path, "raw")
bronze_path: str = os.path.join(hospital_medallion_path, "bronze")
silver_path: str = os.path.join(hospital_medallion_path, "silver")
gold_path: str = os.path.join(hospital_medallion_path, "gold")

# Raw
#TODO
Purpose Input Output Main transformations Validation

In [2]:
#TODO copy input into raw
#TODO print files in raw and prove copied succesffuly


import shutil
import os

os.makedirs(raw_path, exist_ok=True)

for file_name in os.listdir(input_path):
    src: str = os.path.join(input_path, file_name)
    dst: str = os.path.join(raw_path, file_name)
    shutil.copy2(src, dst)

display([f for f in os.listdir(raw_path)])

['hospital_appointments_raw.csv', 'hospital_reference_master.xlsx']

The raw layer should not have business transformations, it is the source data and should stay that way for auditability and repeatability of the pipeline

In [3]:
from pyspark import SparkContext, Broadcast
from pyspark.sql import DataFrame, SparkSession, Column

# Windows-only: Hadoop's local filesystem committer needs winutils.exe/hadoop.dll,
# or writes fail with "HADOOP_HOME and hadoop.home.dir are unset" even for local Parquet.
# Must be set before the SparkSession/JVM gateway starts.
#TODO only do for local
os.environ.setdefault("HADOOP_HOME", "C:\\hadoop")
os.environ["PATH"] = os.environ.get("PATH", "") + ";" + os.path.join(os.environ["HADOOP_HOME"], "bin")

spark: SparkSession = (SparkSession.builder.
                        appName("Hospital Medallion Pipeline")
                        #TODO remove for databricks
                        .master("local[*]")
                       .getOrCreate())
sc: SparkContext = spark.sparkContext

G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


# Bronze
Purpose Input Output Main transformations Validation

In [4]:
#TODO Question 7
#Read hospital_appointments_raw.csv into a Spark DataFrame.

#Use options appropriate for a header-based CSV file.

#Initially load source columns in a way that prevents invalid values from being silently lost.
hospital_appointments_file_name: str = "hospital_appointments_raw.csv"
hospital_appointments_path: str = os.path.join(raw_path, hospital_appointments_file_name)
hospital_appointments_raw_df: DataFrame = spark.read.csv(
    hospital_appointments_path,
    header=True,
    inferSchema=False,
    mode="PERMISSIVE",)

In [5]:
#Read all Excel sheets required for the assignment.

#Create individual DataFrames for:

#doctor_master
#department_targets
#status_mapping
#data_dictionary
#Use a method supported by your workspace. A Python Excel library may be used to read the workbook and then convert each sheet to a Spark DataFrame.

import pandas as pd

hospital_reference_master_path: str = os.path.join(raw_path, "hospital_reference_master.xlsx")

doctor_master_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="doctor_master")
department_targets_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="department_targets")
status_mapping_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="status_mapping")
data_dictionary_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="data_dictionary")

doctor_master_df: DataFrame = spark.createDataFrame(doctor_master_pd)
department_targets_df: DataFrame = spark.createDataFrame(department_targets_pd)
status_mapping_df: DataFrame = spark.createDataFrame(status_mapping_pd)
data_dictionary_df: DataFrame = spark.createDataFrame(data_dictionary_pd)

display({
    "reference_workbook": hospital_reference_master_path,
    "doctor_master_rows": doctor_master_df.count(),
    "department_targets_rows": department_targets_df.count(),
    "status_mapping_rows": status_mapping_df.count(),
    "data_dictionary_rows": data_dictionary_df.count(),
})


G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features 

{'reference_workbook': 'G:\\Owen\\Revature\\Training\\assignments\\week6\\4\\databricks_medallion_hospital_assignment\\Volumes/tables/medallion_hospital\\raw\\hospital_reference_master.xlsx',
 'doctor_master_rows': 10,
 'department_targets_rows': 5,
 'status_mapping_rows': 7,
 'data_dictionary_rows': 17}

In [6]:
#Question 9
#Add the following audit columns to the appointment Bronze DataFrame:

#source_file_name
#source_system_name
#bronze_ingestion_timestamp
#bronze_ingestion_date
#record_hash
import pyspark.sql.functions as F

hospital_appointments_bronze_df: DataFrame = (hospital_appointments_raw_df
                                              .withColumn("source_file_name", F.lit(hospital_appointments_file_name))
                                              .withColumn("source_system_name", F.lit("Hospital Appointment and Revenue Analytics"))
                                              .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
                                              .withColumn("bronze_ingestion_date", F.current_date())
                                              .withColumn("record_hash", F.hash(
                                                  "appointment_id",
                                                    "appointment_date",
                                                    "patient_id",
                                                    "patient_name",
                                                    "age",
                                                    "gender",
                                                    "city",
                                                    "department",
                                                    "doctor_id",
                                                    "doctor_name",
                                                    "appointment_status",
                                                    "consultation_fee",
                                                    "discount_pct",
                                                    "amount_paid",
                                                    "payment_mode",
                                                    "phone_number",
                                                    "source_system",
                                              )))


In [7]:
#Write the appointment Bronze DataFrame and all Excel reference DataFrames to the Bronze layer.

#Suggested names:

#bronze_appointments
#bronze_doctor_master
#bronze_department_targets
#bronze_status_mapping
#bronze_data_dictionary
#Use Delta format when supported. Otherwise, use Parquet and clearly document the choice.
#This workspace (Databricks Legacy Free Edition without Unity Catalog) does not have the delta package
#registered on the Spark classpath, so Parquet is used instead for all Bronze/Silver/Gold writes.


hospital_appointments_bronze_path: str = os.path.join(bronze_path, "bronze_appointments.parquet")
hospital_appointments_bronze_df.write.mode("overwrite").parquet(hospital_appointments_bronze_path)

doctor_master_bronze_path: str = os.path.join(bronze_path, "doctor_master.parquet")
department_targets_bronze_path: str = os.path.join(bronze_path, "department_targets.parquet")
status_mapping_bronze_path: str = os.path.join(bronze_path, "status_mapping.parquet")
data_dictionary_bronze_path: str = os.path.join(bronze_path, "data_dictionary.parquet")

doctor_master_df.write.mode("overwrite").parquet(doctor_master_bronze_path)
department_targets_df.write.mode("overwrite").parquet(department_targets_bronze_path)
status_mapping_df.write.mode("overwrite").parquet(status_mapping_bronze_path)
data_dictionary_df.write.mode("overwrite").parquet(data_dictionary_bronze_path)


In [8]:
#Show:

#Bronze row count
#Distinct appointment ID count
#Exact duplicate count using record_hash
#Duplicate business-key count using appointment_id


appointment_row_count: int = hospital_appointments_bronze_df.count()
distinct_appointment_id_count = (hospital_appointments_bronze_df
                                 .select("appointment_id")
                                 .distinct()
                                 .count())
exact_duplicate_count: int = (hospital_appointments_bronze_df
                         .groupBy("record_hash")
                         .count()
                         .where("count > 1")
                         .count())
duplicate_business_key_count: int = (hospital_appointments_bronze_df
                                .groupBy("appointment_id")
                                .count()
                                .where("count > 1")
                                .count())
print(f"Appointment row count: {appointment_row_count}")
print(f"Distinct appointment ID count: {distinct_appointment_id_count}")
print(f"Exact duplicate count: {exact_duplicate_count}")
print(f"Duplicate business-key count: {duplicate_business_key_count}")


Appointment row count: 60
Distinct appointment ID count: 58
Exact duplicate count: 1
Duplicate business-key count: 2


%md
# Silver
Purpose Input Output Main transformations Validation

In [9]:
# Part D — Silver Data Quality and Transformation
# Perform the following transformations.
#
# Question 13 — Trim and standardize text
# Apply trimming and case standardization to:
#
# patient_name
# city
# department
# doctor_name
# appointment_status
# payment_mode
# source_system
# Expected examples:
#
# "  Diya Sharma  " → "Diya Sharma"
# "chennai" → "Chennai"
# "ORTHOPEDICS" → "Orthopedics"
# "dr. meera iyer" → "Dr. Meera Iyer"

def normalize(column_name: str) -> Column:
    return F.initcap(F.lower(F.trim(F.col(column_name))))


hospital_appointments_silver_df: DataFrame = spark.read.parquet(hospital_appointments_bronze_path)
hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumns({
    "patient_name": normalize("patient_name"),
    "city": normalize("city"),
    "department": normalize("department"),
    "doctor_name": normalize("doctor_name"),
    #appointment status is special
    "appointment_status": F.regexp_replace(F.upper(F.trim(F.col("appointment_status"))), r"_", "_"),
    "payment_mode": normalize("payment_mode"),
    "source_system": normalize("source_system")
}))

In [10]:
#add validation columns

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns({
    #validation will mark this as false if invalid down the road
    "is_valid_record": F.lit(True),
    "validation_error_count": F.lit(0),
    "validation_errors": F.lit("")
})

def validate(validator: Column, invalid_reason: str) -> dict[str, Column]:
    """
    Takes a validator column (a True/False evaluated column), and sets the record to invalid if validator is False and adds the invalid
    :param validator:
    :param invalid_reason:
    :return:
    """
    is_valid: Column = F.when(validator, F.lit(True)).otherwise(F.lit(False))
    return {
        "is_valid_record": F.when(~F.col("is_valid_record"), F.lit(False)).otherwise(is_valid),
        "validation_error_count": F.when(is_valid, F.col("validation_error_count")).otherwise(F.col("validation_error_count") + 1),
        "validation_errors": F.when(is_valid, F.col("validation_errors")).otherwise(F.concat_ws("|", F.col("validation_errors"), F.lit(invalid_reason)))
    }


In [11]:
from typing import Optional

# Question 14 — Standardize appointment status
# Join with status_mapping and derive:
#
# standard_appointment_status
# is_billable
# include_in_utilization
# Unmapped statuses must be marked as rejected or assigned a clear validation failure.

#explicit re-assignment
status_mapping_bronze_df: DataFrame = spark.read.parquet(status_mapping_bronze_path)
hospital_appointments_silver_df = (hospital_appointments_silver_df
 .join(status_mapping_bronze_df,
       on=hospital_appointments_silver_df["appointment_status"] == status_mapping_bronze_df["raw_status"],
       how="left"))

(hospital_appointments_silver_df
 .withColumns(validate(F.col("standard_status").isNotNull(), "INVALID_STATUS")))

def to_boolean(column_name: str) -> Column:
    return F.when(F.upper(F.col(column_name)) == "Y", F.lit(True)).otherwise(F.lit(False))

hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumns({
                                       "standard_appointment_status": F.col("standard_status"),
                                        "is_billable": to_boolean("is_billable"),
                                        "include_in_utilization": to_boolean("include_in_utilization")
                                   }))



In [12]:
# Question 15 — Convert and validate appointment dates
# Convert appointment_date to a valid date.
#
# Support the formats present in the source data.
#
# Create:
#
# appointment_date_clean
# appointment_year
# appointment_month
# appointment_day
# Invalid dates must not be accepted as valid.

hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumn("appointment_date_clean", F.coalesce(
                                       F.try_to_date(F.trim(F.col("appointment_date")), "yyyy-MM-dd"),
                                       F.try_to_date(F.trim(F.col("appointment_date")), "MM/dd/yyyy"),
                                   ))
                                   .withColumns({
                                       "appointment_year": F.year(F.col("appointment_date_clean")),
                                       "appointment_month": F.month(F.col("appointment_date_clean")),
                                       "appointment_day": F.dayofmonth(F.col("appointment_date_clean"))
                                   }))
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(validate(F.col("appointment_date_clean").isNotNull(), "INVALID_DATE"))

In [13]:
# Question 16 — Convert numeric columns
# Convert these columns safely:
#
# age
# consultation_fee
# discount_pct
# amount_paid
# Rows that cannot be converted must be flagged.


hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns({
    "age_raw": F.trim(F.col("age")),
    "consultation_fee_raw": F.trim(F.col("consultation_fee")),
    "discount_pct_raw": F.trim(F.col("discount_pct")),
    "amount_paid_raw": F.trim(F.col("amount_paid")),
})

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns({
    "age": F.col("age_raw").try_cast(IntegerType()),
    "consultation_fee": F.col("consultation_fee_raw").try_cast(DecimalType(10, 2)),
    "discount_pct": F.col("discount_pct_raw").try_cast(DecimalType(5, 2)),
    "amount_paid": F.col("amount_paid_raw").try_cast(DecimalType(10, 2)),
})

def cast_succeeded(raw_column_name: str, cast_column_name: str) -> Column:
    raw_is_blank: Column = F.col(raw_column_name).isNull() | (F.col(raw_column_name) == "")
    return raw_is_blank | F.col(cast_column_name).isNotNull()

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(cast_succeeded("age_raw", "age"), "INVALID_AGE"))
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(cast_succeeded("consultation_fee_raw", "consultation_fee"), "INVALID_FEE"))
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(cast_succeeded("discount_pct_raw", "discount_pct"), "INVALID_DISCOUNT"))
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(cast_succeeded("amount_paid_raw", "amount_paid"), "INVALID_AMOUNT_PAID"))

In [14]:
# Question 17 — Validate age
# Valid age range:
#
# 0 to 110
# Flag negative ages and unrealistic ages.

hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumns(
    validate((F.col("age").between(0, 110)), "INVALID_AGE")
))

In [15]:
# Question 18 — Validate gender
# Accepted values:
#
# M
# F
# O
# Any other value must be rejected or marked invalid.
hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumns(validate(F.col("gender").isin("M", "F", "O"), "INVALID_GENDER")))



In [16]:
# Question 19 — Standardize department values
# Map known aliases where appropriate.
#
# Example:
#
# Cardio → Cardiology
# After standardization, the department must exist in department_targets.

known_aliases: dict[str, str] = {
    "Cardio": "Cardiology",
    "Ortho": "Orthopedics",
    "Neuro": "Neurology"
}
known_aliases_broadcast: Broadcast[dict[str, str]] = sc.broadcast(known_aliases)
#create a column map out of the known aliases
department_alias_map: Column = F.create_map([F.lit(x) for x in sum(known_aliases_broadcast.value.items(), ())])

hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumn("department", F.when(F.col("department").isin(list(known_aliases.keys())),
                                                        department_alias_map[F.col("department")])
                                                        .otherwise(F.col("department"))))

#Explicit re-assignment
department_targets_df: DataFrame = spark.read.parquet(department_targets_bronze_path)

#department_targets is expected to have a low number of rows
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("department").isin([row["department"] for row in department_targets_df.collect()]), "INVALID_DEPARTMENT")
)


In [17]:
# Question 20 — Validate doctor details
# Join the appointment data with doctor_master.
#
# Validate that:
#
# doctor_id exists
# Doctor is active
# Doctor name matches the master
# Doctor belongs to the stated department
# Use the master doctor name and department in the clean output.

doctor_master_df: DataFrame = spark.read.parquet(doctor_master_bronze_path)

hospital_appointments_silver_df = (hospital_appointments_silver_df.alias("appointment")
                                   .join(doctor_master_df.alias("doctor_master"), "doctor_id", "left"))
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn("is_active", to_boolean("doctor_master.active_flag"))

def valid_doctor() -> Column:
    return (F.col("doctor_id").isNotNull() &
            F.col("is_active") &
            (F.col("appointment.doctor_name") == F.col("doctor_master.doctor_name")) &
            (F.col("appointment.department") == F.col("doctor_master.department")))

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(valid_doctor(), "INVALID_DOCTOR")
)

hospital_appointments_silver_df = hospital_appointments_silver_df.drop(
    F.col("doctor_master.doctor_id"), F.col("doctor_master.specialization"), F.col("doctor_master.active_flag"),
    F.col("appointment.doctor_name"), F.col("appointment.department")
)


In [18]:
# Question 21 — Handle duplicate records
# Create rules for:
#
# Exact duplicate records
# Duplicate appointment_id values
# For duplicate appointment IDs, keep one deterministic record and reject the remaining records. Document the ordering rule used.
#
# Ordering rule used for BOTH windows below (deterministic, reproducible across reruns):
#   1. ingestion_timestamp ascending
#   2. record_hash ascending
#   3. _dedup_row_id ascending
# Rank 1 within each partition is kept as the surviving record; every other rank in
# that group is marked invalid.

from pyspark.sql.window import Window, WindowSpec

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn(
    "_dedup_row_id", F.monotonically_increasing_id()
)

exact_duplicate_window: WindowSpec = (Window
                                   .partitionBy("record_hash")
                                   .orderBy(F.col("ingestion_date").asc(),
                                            F.col("record_hash").asc(),
                                            F.col("_dedup_row_id").asc()))

appointment_id_window: WindowSpec = (Window
                                  .partitionBy("appointment_id")
                                  .orderBy(F.col("ingestion_date").asc(),
                                           F.col("record_hash").asc(),
                                           F.col("_dedup_row_id").asc()))

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns({
    "_exact_duplicate_rank": F.row_number().over(exact_duplicate_window),
    "_appointment_id_rank": F.row_number().over(appointment_id_window),
})

# Exact duplicates: content-identical rows (same record_hash). Keep rank 1, reject the rest.
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("_exact_duplicate_rank") == 1, "EXACT_DUPLICATE"))

# Duplicate appointment_id (content may differ): keep rank 1 per the ordering rule above, reject the rest.
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("_appointment_id_rank") == 1, "DUPLICATE_APPOINTMENT_ID"))

# drop helper columns used only for ranking
hospital_appointments_silver_df = hospital_appointments_silver_df.drop(
    "_dedup_row_id", "_exact_duplicate_rank", "_appointment_id_rank"
)


In [19]:
# Question 22 — Handle null values
# Apply appropriate rules:
#
# Missing patient ID → reject
# Missing patient name → reject
# Null discount percentage → treat as 0
# Blank payment mode → allowed only when amount paid is 0
# Missing required master-data match → reject

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("patient_id").isNotNull() & (F.col("patient_id") != ""), "MISSING_PATIENT_ID")
)
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("patient_name").isNotNull() & (F.col("patient_name") != ""), "MISSING_PATIENT_NAME")
)
hospital_appointments_silver_df = hospital_appointments_silver_df.fillna({
    "discount_pct": 0
})
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate((F.col("payment_mode").isNotNull() & (F.col("payment_mode") != "")) | (F.col("amount_paid") == 0), "MISSING_PAYMENT_MODE")
)


In [20]:
# Question 23 — Validate consultation fee
# Rules:
#
# Must be numeric
# Must be greater than or equal to zero

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("consultation_fee") >= 0, "INVALID_FEE")
)

In [21]:
# Question 24 — Validate discount
# Rules:
#
# 0 <= discount_pct <= 100

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.col("discount_pct").between(0, 100), "INVALID_DISCOUNT")
)

In [22]:
# Question 25 — Calculate expected amount
# Create:
#
# expected_amount_paid
# Suggested formula for completed/billable appointments:
#
# consultation_fee × (1 - discount_pct / 100)
# For non-billable statuses, expected paid amount should normally be zero.

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn("expected_amount_paid", F.col("consultation_fee") * (1 - F.col("discount_pct") / 100))


In [23]:
# Question 26 — Validate amount paid
# Flag:
#
# Negative amount paid
# Amount paid much higher than the expected amount
# Completed appointments where actual amount differs from expected amount
# Non-billable appointments with a non-zero payment
# Use a small tolerance for decimal comparison.
high_paid_tolerance: float = 0.15
normal_tolerance: float = 0.01
positive_amount: Column = F.col("amount_paid") > 0
correct_paid: Column = F.col("amount_paid") <= F.col("expected_amount_paid") * (1 + high_paid_tolerance)
completed_correct: Column = F.when(F.col("appointment_status") == "COMPLETED", (F.col("amount_paid") <= F.col("expected_amount_paid") * (1 + normal_tolerance)) & (F.col("amount_paid") >= F.col("expected_amount_paid") * (1 - normal_tolerance))) .otherwise(F.lit(True))
non_billable_zero: Column = F.when(F.col("is_billable") == False, F.col("amount_paid") == 0).otherwise(F.lit(True))

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(positive_amount & correct_paid & completed_correct & non_billable_zero, "INVALID_AMOUNT_PAID")
)


In [24]:
# Question 27 — Validate payment mode
# Accepted paid transaction modes:
#
# UPI
# CARD
# CASH
# NET_BANKING
# Blank is allowed only when no amount was paid.

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn("payment_mode", F.upper(F.col("payment_mode")))

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate((F.col("payment_mode").isin("UPI", "CARD", "CASH", "NET_BANKING", "")) | ((F.col("payment_mode") == "") & (F.col("amount_paid") == 0)), "INVALID_PAYMENT_MODE")
)

In [25]:
# Question 28 — Validate phone number
# A valid phone number must contain exactly 10 numeric digits.
#
# Create a masked phone field for the Silver layer, for example:
#
# 98******21
# Do not expose the full phone number in Gold reports.

hospital_appointments_silver_df = hospital_appointments_silver_df.withColumns(
    validate(F.regexp(F.col("phone_number"), F.lit(r'^\d{10}$')), "INVALID_PHONE")
)

#only show the last 4 numbers
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn(
    "masked_phone_number", F.concat(F.lit("*" * 6), F.substring(F.col("phone_number"), 6 , 4))
)

In [26]:
# Question 29 — Create validation columns
# Create at least:
#
# is_valid_record
# validation_error_count
# validation_errors
# silver_processed_timestamp
# validation_errors should contain one or more readable error reasons.
#
# Example:
#
# INVALID_AGE|INVALID_PHONE
hospital_appointments_silver_df = hospital_appointments_silver_df.withColumn("silver_processed_timestamp", F.current_timestamp())

In [27]:
#Clean leftover columns
#TODO do this is in a better place
hospital_appointments_silver_df = hospital_appointments_silver_df.drop(F.col("raw_status"), F.col("age_raw"), F.col("consultation_fee_raw"), F.col("amount_paid_raw") )

In [28]:
# Question 30 — Split valid and rejected data
# Create:
#
# silver_appointments_clean
# silver_appointments_rejected
# The rejected dataset must retain:
#
# Original identifying columns
# Validation reasons
# Source file
# Bronze ingestion timestamp
# Silver processing timestamp

silver_appointments_clean: DataFrame = hospital_appointments_silver_df.filter(F.col("is_valid_record") == True)
silver_appointments_rejected: DataFrame = hospital_appointments_silver_df.filter(F.col("is_valid_record") == False)


In [29]:
# Question 31 — Silver output
# Write both clean and rejected datasets into the Silver layer.
#
# Display:
#
# Bronze row count
# Clean Silver row count
# Rejected Silver row count
# Reconciliation result
# Required reconciliation:
#
# Bronze count = Clean Silver count + Rejected Silver count

silver_appointments_clean_path: str = os.path.join(silver_path, "silver_appointments_clean.parquet")
silver_appointments_clean.write.mode("overwrite").parquet(silver_appointments_clean_path)
silver_appointments_rejected_path: str = os.path.join(silver_path, "silver_appointments_rejected.parquet")
silver_appointments_rejected.write.mode("overwrite").parquet(silver_appointments_rejected_path)

bronze_count = hospital_appointments_bronze_df.count()
clean_silver_count = silver_appointments_clean.count()
rejected_silver_count = silver_appointments_rejected.count()

print(f"Bronze row count: {bronze_count}")
print(f"Clean Silver row count: {clean_silver_count}")
print(f"Rejected Silver row count: {rejected_silver_count}")
print(f"Reconciliation result: {bronze_count == clean_silver_count + rejected_silver_count}")

Bronze row count: 60
Clean Silver row count: 21
Rejected Silver row count: 39
Reconciliation result: True


# Gold
Purpose Input Output Main transformations Validation

In [30]:
from pyspark.sql.types import StructField, StringType, LongType

# Part E — Gold Layer: Report-Ready Outputs
# Create the following Gold outputs from clean Silver data.
#
# Question 32 — Department monthly performance
# Create a monthly department-level report with:
#
# appointment_year
# appointment_month
# department
# total_appointments
# completed_appointments
# cancelled_appointments
# no_show_appointments
# scheduled_appointments
# completion_rate_pct
# cancellation_rate_pct
# no_show_rate_pct
# gross_consultation_value
# discount_value
# net_revenue
# average_revenue_per_completed_appointment
# unique_patients
# monthly_completed_target
# monthly_revenue_target
# completed_target_achievement_pct
# revenue_target_achievement_pct
# target_status
# Join the report with department_targets.
#
# Suggested target status:
#
# ACHIEVED
# PARTIALLY_ACHIEVED
# NOT_ACHIEVED

#intentional re-assignment
silver_appointments_clean: DataFrame = spark.read.parquet(silver_appointments_clean_path)

# department_targets is read with its native inferred types (no enforced schema-on-read),
# then cast explicitly. The whole-number Excel columns (monthly_completed_target,
# monthly_revenue_target) were written to Bronze as physical Parquet INT64 by
# spark.createDataFrame(pandas_df); pinning a schema with a mismatched physical type
# (IntegerType, or DecimalType for a column with no decimal values) causes a
# PARQUET_COLUMN_DATA_TYPE_MISMATCH failure the first time it's materialized.
department_targets_df: DataFrame = (spark.read.parquet(department_targets_bronze_path)
                                    .withColumns({
                                        "monthly_completed_target": F.col("monthly_completed_target").cast(LongType()),
                                        "monthly_revenue_target": F.col("monthly_revenue_target").cast(DecimalType(10, 2)),
                                        "max_cancel_rate_pct": F.col("max_cancel_rate_pct").cast(DecimalType(5, 2)),
                                    }))
display(silver_appointments_clean)
display(silver_appointments_clean.columns)
display(silver_appointments_clean.schema)
monthly_department_report_gold: DataFrame = silver_appointments_clean.join(department_targets_df, "department")
monthly_department_report_gold = monthly_department_report_gold.groupBy("department", "appointment_year", "appointment_month").agg(
    F.count("appointment_id").alias("total_appointments"),
    F.sum(F.when(F.col("appointment_status") == "COMPLETED", 1).otherwise(0)).alias("completed_appointments"),
    F.sum(F.when(F.col("appointment_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_appointments"),
    F.sum(F.when(F.col("appointment_status") == "NO_SHOW", 1).otherwise(0)).alias("no_show_appointments"),
    F.sum(F.col("consultation_fee")).alias("gross_consultation_value"),
    F.sum(F.col("discount_pct")).alias("discount_value"),
    F.sum(F.col("amount_paid")).alias("net_revenue"),
    F.avg(F.when(F.col("appointment_status") == "COMPLETED", F.col("amount_paid"))).alias("average_revenue_per_completed_appointment"),
    F.countDistinct("patient_id").alias("unique_patients"),
    F.first("monthly_completed_target").alias("monthly_completed_target"),
    F.first("monthly_revenue_target").alias("monthly_revenue_target"),
    F.first("max_cancel_rate_pct").alias("max_cancel_rate_pct"),
)

# rates are derived from the already-aggregated counts, not recomputed with a
# nested aggregate (Spark disallows an aggregate function inside another aggregate's argument)
monthly_department_report_gold = monthly_department_report_gold.withColumns({
    "completion_rate_pct": (F.col("completed_appointments") / F.col("total_appointments")) * 100,
    "cancellation_rate_pct": (F.col("cancelled_appointments") / F.col("total_appointments")) * 100,
    "no_show_rate_pct": (F.col("no_show_appointments") / F.col("total_appointments")) * 100,
})

monthly_department_report_gold = monthly_department_report_gold.withColumns({
    "completed_target_achievement_pct": (F.col("completed_appointments") / F.col("monthly_completed_target")) * 100,
    "revenue_target_achievement_pct": (F.col("net_revenue") / F.col("monthly_revenue_target")) * 100,
})

completed_achieved: Column = F.col("completed_target_achievement_pct") >= 100
revenue_achieved: Column = F.col("revenue_target_achievement_pct") >= 100
cancel_rate_achieved: Column = F.col("cancellation_rate_pct") <= F.col("max_cancel_rate_pct")

achieved_count: Column = (
  completed_achieved.cast(IntegerType())
  + revenue_achieved.cast(DoubleType())
  + cancel_rate_achieved.cast(DoubleType())
)

monthly_department_report_gold = monthly_department_report_gold.withColumn(
  "target_status",
  F.when(achieved_count == 3, F.lit("ACHIEVED"))
   .when(achieved_count >= 1, F.lit("PARTIALLY_ACHIEVED"))
   .otherwise(F.lit("NOT_ACHIEVED"))
)


DataFrame[doctor_id: string, appointment_id: string, appointment_date: string, patient_id: string, patient_name: string, age: int, gender: string, city: string, appointment_status: string, consultation_fee: decimal(10,2), discount_pct: decimal(5,2), amount_paid: decimal(10,2), payment_mode: string, phone_number: string, source_system: string, ingestion_date: string, source_file_name: string, source_system_name: string, bronze_ingestion_timestamp: timestamp, bronze_ingestion_date: date, record_hash: int, is_valid_record: boolean, validation_error_count: int, validation_errors: string, standard_status: string, is_billable: boolean, include_in_utilization: boolean, standard_appointment_status: string, appointment_date_clean: date, appointment_year: int, appointment_month: int, appointment_day: int, discount_pct_raw: string, doctor_name: string, department: string, is_active: boolean, expected_amount_paid: decimal(21,8), masked_phone_number: string, silver_processed_timestamp: timestamp]

['doctor_id',
 'appointment_id',
 'appointment_date',
 'patient_id',
 'patient_name',
 'age',
 'gender',
 'city',
 'appointment_status',
 'consultation_fee',
 'discount_pct',
 'amount_paid',
 'payment_mode',
 'phone_number',
 'source_system',
 'ingestion_date',
 'source_file_name',
 'source_system_name',
 'bronze_ingestion_timestamp',
 'bronze_ingestion_date',
 'record_hash',
 'is_valid_record',
 'validation_error_count',
 'validation_errors',
 'standard_status',
 'is_billable',
 'include_in_utilization',
 'standard_appointment_status',
 'appointment_date_clean',
 'appointment_year',
 'appointment_month',
 'appointment_day',
 'discount_pct_raw',
 'doctor_name',
 'department',
 'is_active',
 'expected_amount_paid',
 'masked_phone_number',
 'silver_processed_timestamp']

StructType([StructField('doctor_id', StringType(), True), StructField('appointment_id', StringType(), True), StructField('appointment_date', StringType(), True), StructField('patient_id', StringType(), True), StructField('patient_name', StringType(), True), StructField('age', IntegerType(), True), StructField('gender', StringType(), True), StructField('city', StringType(), True), StructField('appointment_status', StringType(), True), StructField('consultation_fee', DecimalType(10,2), True), StructField('discount_pct', DecimalType(5,2), True), StructField('amount_paid', DecimalType(10,2), True), StructField('payment_mode', StringType(), True), StructField('phone_number', StringType(), True), StructField('source_system', StringType(), True), StructField('ingestion_date', StringType(), True), StructField('source_file_name', StringType(), True), StructField('source_system_name', StringType(), True), StructField('bronze_ingestion_timestamp', TimestampType(), True), StructField('bronze_inges

In [31]:
# Question 33 — Doctor performance report
# Create a doctor-level report with:
#
# doctor_id
# doctor_name
# department
# specialization
# total_appointments
# completed_appointments
# cancelled_appointments
# no_show_appointments
# completion_rate_pct
# no_show_rate_pct
# unique_patients
# net_revenue
# average_revenue_per_completed_appointment
# Rank doctors within each department by:
#
# Net revenue
# Completed appointments

doctor_master_structure: StructType = StructType([
      StructField("doctor_id", StringType(), nullable=False),
      StructField("doctor_name", StringType(), nullable=False),
      StructField("department", StringType(), nullable=False),
      StructField("specialization", StringType(), nullable=False),
    StructField("active_flag", StringType(), nullable=False),
  ])

#intentional re-assignment
doctor_master_bronze: DataFrame = spark.read.schema(doctor_master_structure).parquet(doctor_master_bronze_path)

gold_doctor_performance_report: DataFrame = silver_appointments_clean.alias("appointment").join(doctor_master_bronze.alias("doctor_master"), on="doctor_id", how="left")
gold_doctor_performance_report = gold_doctor_performance_report.groupBy(
    F.col("appointment.doctor_id"),
    F.col("appointment.doctor_name"),
    F.col("doctor_master.department"),
    F.col("doctor_master.specialization"),
).agg(
    F.count(F.col("appointment.appointment_id")).alias("total_appointments"),
    F.sum(F.when(F.col("appointment.appointment_status") == "COMPLETED", 1).otherwise(0)).alias("completed_appointments"),
    F.sum(F.when(F.col("appointment.appointment_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_appointments"),
    F.sum(F.when(F.col("appointment.appointment_status") == "NO_SHOW", 1).otherwise(0)).alias("no_show_appointments"),
    F.countDistinct(F.col("appointment.patient_id")).alias("unique_patients"),
    F.sum(F.col("appointment.amount_paid")).alias("net_revenue"),
    F.avg(F.when(F.col("appointment.appointment_status") == "COMPLETED", F.col("appointment.amount_paid"))).alias("average_revenue_per_completed_appointment"),
)

# rates are derived from the already-aggregated counts, not recomputed with a
# nested aggregate (Spark disallows an aggregate function inside another aggregate's argument)
gold_doctor_performance_report = gold_doctor_performance_report.withColumns({
    "completion_rate_pct": (F.col("completed_appointments") / F.col("total_appointments")) * 100,
    "no_show_rate_pct": (F.col("no_show_appointments") / F.col("total_appointments")) * 100,
})

# Rank doctors within each department by net revenue, then by completed appointments.
department_revenue_window: WindowSpec = Window.partitionBy("department").orderBy(F.col("net_revenue").desc())
department_completed_window: WindowSpec = Window.partitionBy("department").orderBy(F.col("completed_appointments").desc())

gold_doctor_performance_report = gold_doctor_performance_report.withColumns({
    "department_revenue_rank": F.rank().over(department_revenue_window),
    "department_completed_appointments_rank": F.rank().over(department_completed_window),
})


In [32]:
# Question 34 — Daily operational trend
# Create a daily report:
#
# appointment_date_clean
# department
# total_appointments
# completed_appointments
# cancelled_appointments
# no_show_appointments
# net_revenue

gold_daily_operational_trend: DataFrame = silver_appointments_clean.groupBy("appointment_date_clean", "department").agg(
    F.count("appointment_id").alias("total_appointments"),
    F.sum(F.when(F.col("appointment_status") == "COMPLETED", 1).otherwise(0)).alias("completed_appointments"),
    F.sum(F.when(F.col("appointment_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_appointments"),
    F.sum(F.when(F.col("appointment_status") == "NO_SHOW", 1).otherwise(0)).alias("no_show_appointments"),
    F.sum(F.col("amount_paid")).alias("net_revenue"),
)


In [33]:
# Question 35 — Source-system performance
# Create a source-system report:
#
# source_system
# total_appointments
# completed_appointments
# conversion_to_completed_pct
# cancelled_appointments
# no_show_appointments
# net_revenue
# average_revenue

gold_source_system_report: DataFrame = silver_appointments_clean.groupBy("source_system").agg(
    F.count("appointment_id").alias("total_appointments"),
    F.sum(F.when(F.col("appointment_status") == "COMPLETED", 1).otherwise(0)).alias("completed_appointments"),
    F.sum(F.when(F.col("appointment_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_appointments"),
    F.sum(F.when(F.col("appointment_status") == "NO_SHOW", 1).otherwise(0)).alias("no_show_appointments"),
    F.sum(F.col("amount_paid")).alias("net_revenue"),
    F.avg(F.col("amount_paid")).alias("average_revenue"),
)

# rate is derived from the already-aggregated counts, not recomputed with a
# nested aggregate (Spark disallows an aggregate function inside another aggregate's argument)
gold_source_system_report = gold_source_system_report.withColumn(
    "conversion_to_completed_pct", (F.col("completed_appointments") / F.col("total_appointments")) * 100
)


In [34]:
# Question 36 — Data-quality summary
# Create a Gold quality report containing:
#
# quality_rule
# failed_record_count
# failed_record_pct
# Include at least these quality categories:
#
# Duplicate records
# Invalid date
# Missing patient
# Invalid age
# Invalid gender
# Invalid department
# Invalid doctor
# Invalid fee
# Invalid discount
# Invalid amount paid
# Invalid payment mode
# Invalid phone
# Invalid status

total_processed_count: int = hospital_appointments_bronze_df.count()

gold_data_quality_report: DataFrame = (silver_appointments_rejected
                                       .withColumn("validation_error", F.explode(F.split(F.col("validation_errors"), r"\|")))
                                       .filter(F.col("validation_error") != ""))

gold_data_quality_report = gold_data_quality_report.groupBy("quality_rule").agg(
    F.count(F.lit(1)).alias("failed_record_count")
)
gold_data_quality_report = gold_data_quality_report.withColumn(
    "failed_record_pct", (F.col("failed_record_count") / F.lit(total_processed_count)) * 100
)


In [35]:
#Write all Gold outputs to the Gold layer.

#Suggested names:

#gold_monthly_department_report
#gold_doctor_performance_report
#gold_daily_operational_trend
#gold_source_system_report
#gold_data_quality_report
#Parquet is used instead of Delta for the same reason as Bronze/Silver: no delta package
#registered on the Spark classpath in this workspace.

gold_monthly_department_report_path: str = os.path.join(gold_path, "gold_monthly_department_report.parquet")
monthly_department_report_gold.write.mode("overwrite").parquet(gold_monthly_department_report_path)

gold_doctor_performance_report_path: str = os.path.join(gold_path, "gold_doctor_performance_report.parquet")
gold_doctor_performance_report.write.mode("overwrite").parquet(gold_doctor_performance_report_path)

gold_daily_operational_trend_path: str = os.path.join(gold_path, "gold_daily_operational_trend.parquet")
gold_daily_operational_trend.write.mode("overwrite").parquet(gold_daily_operational_trend_path)

gold_source_system_report_path: str = os.path.join(gold_path, "gold_source_system_report.parquet")
gold_source_system_report.write.mode("overwrite").parquet(gold_source_system_report_path)

gold_data_quality_report_path: str = os.path.join(gold_path, "gold_data_quality_report.parquet")
gold_data_quality_report.write.mode("overwrite").parquet(gold_data_quality_report_path)

print(f"Monthly department report rows: {monthly_department_report_gold.count()}")
print(f"Doctor performance report rows: {gold_doctor_performance_report.count()}")
print(f"Daily operational trend rows: {gold_daily_operational_trend.count()}")
print(f"Source-system report rows: {gold_source_system_report.count()}")
print(f"Data-quality report rows: {gold_data_quality_report.count()}")


Monthly department report rows: 10
Doctor performance report rows: 9
Daily operational trend rows: 21
Source-system report rows: 3
Data-quality report rows: 11


In [36]:
# Question 37 — Top business insights
# Using the Gold outputs, answer these questions:
#
# Which department generated the highest net revenue?
# Which department had the highest cancellation rate?
# Which department had the highest no-show rate?
# Which doctor completed the most appointments?
# Which doctor generated the highest revenue?
# Which source system produced the highest completed conversion rate?
# Which month generated the highest revenue?
# Which department missed its monthly target by the largest percentage?
# What are the three most common data-quality failures?
# How many records were rejected from the source?

print("Which department generated the highest net revenue?", monthly_department_report_gold
      .groupBy("department")
      .agg(F.sum(F.col("net_revenue")).alias("total_net_revenue"))
      .orderBy(F.desc("total_net_revenue"))
      .select("department")
      .first()["department"])

print("Which department had the highest cancellation rate?", monthly_department_report_gold
      .groupBy("department")
      .agg(F.avg(F.col("cancellation_rate_pct")).alias("cancellation_rate_pct"))
      .orderBy(F.desc("cancellation_rate_pct"))
      .select("department")
      .first()["department"])

print("Which department had the highest no-show rate?", monthly_department_report_gold
      .groupBy("department")
      .agg(F.avg(F.col("no_show_rate_pct")).alias("no_show_rate_pct"))
      .orderBy(F.desc("no_show_rate_pct"))
      .select("department")
      .first()["department"])

print("Which doctor completed the most appointments?", gold_doctor_performance_report
      .orderBy(F.desc("completed_appointments"))
      .select("doctor_name")
      .first()["doctor_name"])

print("Which doctor generated the highest revenue?", gold_doctor_performance_report
      .orderBy(F.desc("net_revenue"))
      .select("doctor_name")
      .first()["doctor_name"])

print("Which source system produced the highest completed conversion rate?", gold_source_system_report
      .orderBy(F.desc("conversion_to_completed_pct"))
      .select("source_system")
      .first()["source_system"])

print("Which month generated the highest revenue?", monthly_department_report_gold
      .groupBy("appointment_month", "appointment_year")
      .agg(F.sum(F.col("net_revenue")).alias("total_net_revenue"))
      .orderBy(F.desc("total_net_revenue"))
      .select("appointment_month", "appointment_year")
      .first())

print("Which department missed its monthly target by the largest percentage?", monthly_department_report_gold
      .withColumn("missed_target_pct", 100 - F.col("completed_target_achievement_pct"))
      .orderBy(F.desc("missed_target_pct"))
      .select("department")
      .first()["department"])

print("What are the three most common data-quality failures?", gold_data_quality_report
      .orderBy(F.desc("failed_record_count"))
      .select("quality_rule")
      .take(3))

print("How many records were rejected from the source?", silver_appointments_rejected.count())



Which department generated the highest net revenue? Dermatology
Which department had the highest cancellation rate? Neurology
Which department had the highest no-show rate? Neurology
Which doctor completed the most appointments? Dr. Ananya Rao
Which doctor generated the highest revenue? Dr. Ananya Rao
Which source system produced the highest completed conversion rate? Call_center
Which month generated the highest revenue? Row(appointment_month=3, appointment_year=2026)
Which department missed its monthly target by the largest percentage? Orthopedics
What are the three most common data-quality failures? [Row(quality_rule='Invalid amount paid'), Row(quality_rule='Invalid payment mode'), Row(quality_rule='Invalid fee')]
How many records were rejected from the source? 39
